<a href="https://colab.research.google.com/github/humaaslam46/-Personal-Blog-Website/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humaaslam46/Internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Random Forest.** My ML-07 baseline rule combines two signals (staleness x CTR-vs-tier)
with a fixed formula - it can't learn how much each signal actually matters, or find
combinations I didn't think to encode by hand. A Random Forest can weigh 8 features
against each other and capture non-linear interactions, while staying robust to the
noisy/small-n buckets I found in ML-07 (e.g. staleness's mixed pattern past 180 days).
I chose Random Forest over plain Logistic Regression because ML-01 already showed
tree-based ensembles outperforming simpler models on this same kind of task.

In [1]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("Internship-ml"):
        os.system("git clone --depth 1 https://github.com/humaaslam46/Internship-ml")
    os.chdir("Internship-ml")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
print("Ready:", os.getcwd())

Ready: /content/Internship-ml


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

**Grouped split by client_id (75/25), not a random row split.** A random split would let
one client's other pages leak into training while testing on that same client - since
pages from the same client share systematic traits (industry, content style, baseline
traffic), that would inflate the score without proving real generalization. GroupShuffleSplit
guarantees zero client overlap between train and test.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

print("Train rows:", len(train), "| Test rows:", len(test))
print("Train clients:", train["client_id"].nunique(), "| Test clients:", test["client_id"].nunique())
print("Client overlap (must be 0):", len(set(train["client_id"]) & set(test["client_id"])))

NameError: name 'df' is not defined

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# rebuild the exact ML-07 baseline so it's scored on this same test set
tier_avg = df.groupby("position_tier")["ctr"].transform("mean")
df["ctr_vs_tier_avg"] = df["ctr"] / tier_avg.replace(0, 0.0001)
df["is_moderately_stale"] = df["days_since_last_update"].between(31, 180).astype(int)
df["baseline_action_score"] = df["impressions_90d"] * df["is_moderately_stale"] * (1 / df["ctr_vs_tier_avg"].clip(lower=0.05))
test["baseline_action_score"] = df.loc[test.index, "baseline_action_score"]

baseline_p50 = precision_at_k(test["baseline_action_score"], test["is_declining"], 50)

features = ["impressions_90d", "days_since_last_update", "avg_position", "ctr",
            "word_count", "engagement_rate", "content_age_days", "search_volume"]
Xtr, ytr = train[features].replace([np.inf, -np.inf], np.nan).fillna(0), train["is_declining"]
Xte, yte = test[features].replace([np.inf, -np.inf], np.nan).fillna(0), test["is_declining"]

rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
rf_p50 = precision_at_k(rf.predict_proba(Xte)[:, 1], yte.values, 50)

print(f"Baseline rule  Precision@50: {baseline_p50:.3f}")
print(f"Random Forest  Precision@50: {rf_p50:.3f}")
print(f"Improvement: {rf_p50/baseline_p50:.2f}x")

## 4. Errors and interpretation

impressions_90d and content_age_days dominate - staleness (days_since_last_update)
actually has *negative* importance, confirming ML-07's MIXED verdict: it's not a
reliable signal on its own.

False positives (top 50, not actually declining): all have ctr=0.0 but small-to-moderate
impressions - the model over-weights "zero CTR" as a red flag, but some zero-CTR pages
are simply new or too low-volume to have earned clicks yet, not truly declining.

False negatives (real decliners scored lowest): also ctr=0.0, but with very low
impressions_90d (1-665) - the model deprioritizes low-visibility pages even when they
are genuinely declining, since low volume looks unremarkable regardless of trend.

Net: the model is better at ranking *visible* pages correctly, but both its false
positives and false negatives cluster around the same blind spot - very low-traffic,
zero-CTR pages, where the signal is genuinely thin either way.

In [ ]:
from sklearn.inspection import permutation_importance
perm = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=42, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=features).sort_values(ascending=False)
print(imp.round(4))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.